# Atelier 1 — Explorer les sources de données
## Projet fil rouge : BigData Commerce Analytics

**Niveau :** B2 · **Modalité :** binôme · 

### Mission
Vous êtes une équipe Data chargée d'étudier les sources d'une plateforme e-commerce de démonstration. Votre rôle n'est pas encore de construire un modèle : vous devez comprendre quelles données existent, qui les produit, comment elles sont structurées, quelles limites elles présentent et quelle valeur métier elles peuvent apporter.

### Compétences visées
- Interroger une API REST et comprendre une réponse JSON.
- Distinguer source, ressource, enregistrement et format.
- Inventorier les sources et leurs métadonnées.
- Justifier qualitativement les 5V.
- Identifier les usages métier et les risques de qualité.

> **Précaution de méthode :** DummyJSON fournit des données fictives/de démonstration. Les paniers ne constituent pas un historique de ventes réel. Les ressources et leurs volumes peuvent évoluer. Les résultats de votre collecte sont un instantané.


## 0. Préparer l'environnement

Exécutez cette cellule. Si `requests` n'est pas installé, lancez dans un terminal : `pip install requests pandas matplotlib`.


In [1]:
from pathlib import Path
import json
import requests
import pandas as pd

BASE = "https://dummyjson.com"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
print("Dossier projet :", ROOT)
print("Dossier données brutes :", RAW)


Dossier projet : C:\Users\Asus\Documents\cours 2026-2027\BigData_Commerce_Analytics_DummyJSON
Dossier données brutes : C:\Users\Asus\Documents\cours 2026-2027\BigData_Commerce_Analytics_DummyJSON\data\raw


## 1. Comprendre une API REST

Une API expose des ressources accessibles par des URL. Une requête GET demande une représentation de la ressource. DummyJSON renvoie généralement un objet JSON contenant une liste, un total, un décalage (`skip`) et une limite (`limit`).

Exemple : `/products?limit=0` demande tous les produits disponibles au moment de l'appel.


In [2]:
from pathlib import Path
import json, requests, pandas as pd

BASE = "https://dummyjson.com"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

url = f"{BASE}/products?limit=5"
response = requests.get(url, timeout=30)
response.raise_for_status()
payload = response.json()
print("Code HTTP :", response.status_code)
print("Clés de premier niveau :", list(payload.keys()))
print("Total annoncé par l'API :", payload.get("total"))
print("Nombre reçu :", len(payload.get("products", [])))
display(pd.json_normalize(payload["products"]).head())


Code HTTP : 200
Clés de premier niveau : ['products', 'total', 'skip', 'limit']
Total annoncé par l'API : 194
Nombre reçu : 5


,id,title,description,category,price,discountPercentage,rating,stock,tags,brand,...,minimumOrderQuantity,images,thumbnail,dimensions.width,dimensions.height,dimensions.depth,meta.createdAt,meta.updatedAt,meta.barcode,meta.qrCode
0,1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular...,beauty,9.99,10.48,2.56,99,"[beauty, mascara]",Essence,...,48,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,15.14,13.08,22.99,2025-10-09T14:47:01.588Z,2026-05-23T11:27:41.868Z,5784719087687,https://cdn.dummyjson.com/public/qr-code.png
1,2,Eyeshadow Palette with Mirror,The Eyeshadow Palette with Mirror offers a ver...,beauty,19.99,18.19,2.86,34,"[beauty, eyeshadow]",Glamour Beauty,...,20,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,9.26,22.47,27.67,2024-12-03T11:29:45.706Z,2025-08-24T16:52:47.806Z,9170275171413,https://cdn.dummyjson.com/public/qr-code.png
2,3,Powder Canister,The Powder Canister is a finely milled setting...,beauty,14.99,9.84,4.64,89,"[beauty, face powder]",Velvet Touch,...,22,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,29.27,27.93,20.59,2025-01-08T18:30:12.844Z,2025-11-26T14:49:26.980Z,8418883906837,https://cdn.dummyjson.com/public/qr-code.png
3,4,Red Lipstick,The Red Lipstick is a classic and bold choice ...,beauty,12.99,12.16,4.36,91,"[beauty, lipstick]",Chic Cosmetics,...,40,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,18.11,28.38,22.17,2025-11-22T23:28:00.911Z,2026-01-26T01:31:40.721Z,9467746727219,https://cdn.dummyjson.com/public/qr-code.png
4,5,Red Nail Polish,The Red Nail Polish offers a rich and glossy r...,beauty,8.99,11.44,4.32,79,"[beauty, nail polish]",Nail Couture,...,22,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,21.63,16.48,29.84,2025-10-03T07:39:53.470Z,2026-04-24T00:21:44.635Z,4063010628104,https://cdn.dummyjson.com/public/qr-code.png


### Questions de compréhension
1. Que signifie le code HTTP 200 ?
2. Pourquoi l'API renvoie-t-elle `total`, `skip` et `limit` ?
3. Quelle différence faites-vous entre une API et un fichier CSV ?
4. Pourquoi faut-il conserver la date de collecte d'un instantané ?


In [3]:
reponses_api = {
    "http_200": "Le code 200 signifie que la requête a réussi et que le serveur a renvoyé une réponse.",
    "pagination": "total indique le nombre total d'éléments disponibles; limit fixe le nombre demandé et skip le nombre d'éléments ignorés. Ces paramètres permettent de récupérer les résultats par pages.",
    "api_vs_csv": "Une API fournit des données à la demande dans une réponse structurée (ici du JSON); un CSV est un fichier tabulaire déjà stocké, lisible sans interroger un serveur.",
    "date_collecte": "La date permet de savoir quand l'instantané a été obtenu, de comparer des collectes et de tenir compte des changements possibles de l'API."
}
reponses_api

{'http_200': 'Le code 200 signifie que la requête a réussi et que le serveur a renvoyé une réponse.',
 'pagination': "total indique le nombre total d'éléments disponibles; limit fixe le nombre demandé et skip le nombre d'éléments ignorés. Ces paramètres permettent de récupérer les résultats par pages.",
 'api_vs_csv': 'Une API fournit des données à la demande dans une réponse structurée (ici du JSON); un CSV est un fichier tabulaire déjà stocké, lisible sans interroger un serveur.',
 'date_collecte': "La date permet de savoir quand l'instantané a été obtenu, de comparer des collectes et de tenir compte des changements possibles de l'API."}

## 2. Inventorier les ressources disponibles

Le script de collecte du kit interroge plusieurs ressources : produits, paniers, utilisateurs, publications, commentaires, recettes, citations et tâches. Les trois premières sont centrales pour le scénario e-commerce ; les autres permettent d'élargir la réflexion sur la variété des données.


In [4]:
resources = ["products", "carts", "users", "posts", "comments", "recipes", "quotes", "todos"]
inventaire = []
mode_donnees = "API en direct"
for resource in resources:
    try:
        r = requests.get(f"{BASE}/{resource}?limit=0", timeout=15)
        r.raise_for_status()
        p = r.json()
        count = len(p.get(resource, [])) if isinstance(p, dict) else len(p)
        inventaire.append({"ressource":resource, "mode":mode_donnees, "code_http":r.status_code,
                           "total_api":p.get("total", count), "elements_reçus":count,
                           "clés_réponse":", ".join(p.keys()) if isinstance(p, dict) else "liste"})
    except Exception:
        # Repli sur l'échantillon local fourni dans data/raw/
        mode_donnees = "échantillon local hors ligne"
        local_file = RAW / f"{resource}.json"
        if local_file.exists():
            p = json.loads(local_file.read_text(encoding="utf-8"))
            count = len(p.get(resource, [])) if isinstance(p, dict) else len(p)
            inventaire.append({"ressource":resource, "mode":mode_donnees, "code_http":"local",
                               "total_api":p.get("total", count), "elements_reçus":count,
                               "clés_réponse":", ".join(p.keys()) if isinstance(p, dict) else "liste"})
        else:
            inventaire.append({"ressource":resource, "mode":"indisponible", "code_http":"ERREUR",
                               "total_api":None, "elements_reçus":None, "clés_réponse":"pas de fichier local"})
inventaire_df = pd.DataFrame(inventaire)
display(inventaire_df)
print("Attention : si le mode local est utilisé, les volumes sont ceux du mini-échantillon pédagogique.")


,ressource,mode,code_http,total_api,elements_reçus,clés_réponse
0,products,API en direct,200,194,194,"products, total, skip, limit"
1,carts,API en direct,200,208,208,"carts, total, skip, limit"
2,users,API en direct,200,208,208,"users, total, skip, limit"
3,posts,API en direct,200,251,251,"posts, total, skip, limit"
4,comments,API en direct,200,340,340,"comments, total, skip, limit"
5,recipes,API en direct,200,50,50,"recipes, total, skip, limit"
6,quotes,API en direct,200,1454,1454,"quotes, total, skip, limit"
7,todos,API en direct,200,254,254,"todos, total, skip, limit"


Attention : si le mode local est utilisé, les volumes sont ceux du mini-échantillon pédagogique.


In [5]:
# Réponses au travail demandé
ressources_ecommerce = ["products", "carts", "users"]
ressource_max = inventaire_df.loc[
    pd.to_numeric(inventaire_df["elements_reçus"], errors="coerce").idxmax(), "ressource"
]
reponses_inventaire = {
    "trois_ressources_utiles": "products (catalogue/prix), carts (articles et quantités des paniers), users (informations de profil pour une segmentation pédagogique).",
    "posts_et_comments": "Les posts sont des publications et les comments leur sont associés via un post; ils ne décrivent ni achat, ni paiement, ni commande. Les commentaires ne sont pas directement reliés aux produits.",
    "ressource_plus_volumineuse": f"{ressource_max}, selon le nombre d'éléments observé dans le tableau d'inventaire.",
    "date_et_heure": "Voir la cellule de collecte ci-dessous; elle affiche l'heure UTC au moment de son exécution."
}
reponses_inventaire

{'trois_ressources_utiles': 'products (catalogue/prix), carts (articles et quantités des paniers), users (informations de profil pour une segmentation pédagogique).',
 'posts_et_comments': 'Les posts sont des publications et les comments leur sont associés via un post; ils ne décrivent ni achat, ni paiement, ni commande. Les commentaires ne sont pas directement reliés aux produits.',
 'ressource_plus_volumineuse': "quotes, selon le nombre d'éléments observé dans le tableau d'inventaire.",
 'date_et_heure': "Voir la cellule de collecte ci-dessous; elle affiche l'heure UTC au moment de son exécution."}

### Travail demandé
- Relevez les trois ressources les plus utiles pour analyser l'activité e-commerce.
- Expliquez pourquoi `posts` et `comments` ne sont pas des commandes.
- Comparez le volume des ressources : laquelle est la plus volumineuse dans votre collecte ?
- Notez la date et l'heure de votre collecte.


In [6]:
from datetime import datetime, timezone
date_collecte = datetime.now(timezone.utc).isoformat(timespec="seconds")
print("Collecte effectuée le :", date_collecte)

print("Réponses au travail demandé :")
print("Ressources e-commerce : products, carts et users.")
print("posts/comments : contenu éditorial, pas des transactions.")
print("Collecte UTC :", date_collecte)


Collecte effectuée le : 2026-11-25T12:43:01+00:00
Réponses au travail demandé :
Ressources e-commerce : products, carts et users.
posts/comments : contenu éditorial, pas des transactions.
Collecte UTC : 2026-11-25T12:43:01+00:00


## 3. Examiner la structure des données

Une donnée JSON peut contenir des objets imbriqués et des tableaux. Cette variété est importante : les informations d'un panier, par exemple, sont imbriquées dans une liste `products`.


In [7]:
def charger_ressource(resource):
    try:
        r = requests.get(f"{BASE}/{resource}?limit=0", timeout=15)
        r.raise_for_status()
        return r.json()
    except Exception:
        local_file = RAW / f"{resource}.json"
        if not local_file.exists():
            raise FileNotFoundError(f"API inaccessible et fichier local absent : {local_file}")
        print(f"API inaccessible : utilisation de l'échantillon local {resource}.json")
        return json.loads(local_file.read_text(encoding="utf-8"))

products_payload = charger_ressource("products")
carts_payload = charger_ressource("carts")
users_payload = charger_ressource("users")

products_df = pd.json_normalize(products_payload["products"])
carts_df = pd.json_normalize(carts_payload["carts"])
users_df = pd.json_normalize(users_payload["users"])

print("Produits :", products_df.shape)
print("Paniers :", carts_df.shape)
print("Utilisateurs :", users_df.shape)
display(products_df.head(3))
display(carts_df.head(3))
display(users_df.head(3))

reponses_analyse = {
    "identifiant_produit": "Le champ id identifie chaque produit de manière unique dans cette ressource.",
    "lien_panier_utilisateur": "Le champ userId du panier référence l'identifiant id de l'utilisateur.",
    "produits_imbriques": "Un panier contient une liste de produits; un panier peut donc produire plusieurs lignes. Il faut aplatir cette liste pour analyser chaque article sans perdre le lien avec le panier.",
    "json_normalize": "json_normalize transforme des objets JSON, y compris leurs champs imbriqués, en colonnes tabulaires exploitables avec Pandas."
}
print("\\nRéponses aux questions d'analyse :")
for question, reponse in reponses_analyse.items():
    print(f"- {question} : {reponse}")


Produits : (194, 27)
Paniers : (208, 7)
Utilisateurs : (208, 52)


,id,title,description,category,price,discountPercentage,rating,stock,tags,brand,...,minimumOrderQuantity,images,thumbnail,dimensions.width,dimensions.height,dimensions.depth,meta.createdAt,meta.updatedAt,meta.barcode,meta.qrCode
0,1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular...,beauty,9.99,10.48,2.56,99,"[beauty, mascara]",Essence,...,48,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,15.14,13.08,22.99,2025-10-09T14:47:01.588Z,2026-05-23T11:27:41.868Z,5784719087687,https://cdn.dummyjson.com/public/qr-code.png
1,2,Eyeshadow Palette with Mirror,The Eyeshadow Palette with Mirror offers a ver...,beauty,19.99,18.19,2.86,34,"[beauty, eyeshadow]",Glamour Beauty,...,20,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,9.26,22.47,27.67,2024-12-03T11:29:45.706Z,2025-08-24T16:52:47.806Z,9170275171413,https://cdn.dummyjson.com/public/qr-code.png
2,3,Powder Canister,The Powder Canister is a finely milled setting...,beauty,14.99,9.84,4.64,89,"[beauty, face powder]",Velvet Touch,...,22,[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...,29.27,27.93,20.59,2025-01-08T18:30:12.844Z,2025-11-26T14:49:26.980Z,8418883906837,https://cdn.dummyjson.com/public/qr-code.png


,id,products,total,discountedTotal,userId,totalProducts,totalQuantity
0,1,"[{'id': 162, 'title': 'Blue Frock', 'price': 2...",13037.88,11510.81,1,4,12
1,2,"[{'id': 86, 'title': 'Man Short Sleeve Shirt',...",139.93,125.70,2,2,7
2,3,"[{'id': 24, 'title': 'Fish Steak', 'price': 14...",1794.85,1625.77,3,6,15


,id,firstName,lastName,maidenName,age,gender,email,phone,username,password,...,company.address.city,company.address.state,company.address.stateCode,company.address.postalCode,company.address.coordinates.lat,company.address.coordinates.lng,company.address.country,crypto.coin,crypto.wallet,crypto.network
0,1,Emily,Johnson,Smith,29,female,emily.johnson@x.dummyjson.com,+81 965-431-3024,emilys,emilyspass,...,San Francisco,Wisconsin,WI,37657,71.814525,-161.150263,United States,Bitcoin,0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a,Ethereum (ERC20)
1,2,Michael,Williams,,36,male,michael.williams@x.dummyjson.com,+49 258-627-6644,michaelw,michaelwpass,...,Los Angeles,New Hampshire,NH,73442,79.098326,-119.624845,United States,Bitcoin,0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a,Ethereum (ERC20)
2,3,Sophia,Brown,,43,female,sophia.brown@x.dummyjson.com,+81 210-652-2785,sophiab,sophiabpass,...,Dallas,Nevada,NV,88511,20.086743,-34.577107,United States,Bitcoin,0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a,Ethereum (ERC20)


\nRéponses aux questions d'analyse :
- identifiant_produit : Le champ id identifie chaque produit de manière unique dans cette ressource.
- lien_panier_utilisateur : Le champ userId du panier référence l'identifiant id de l'utilisateur.
- produits_imbriques : Un panier contient une liste de produits; un panier peut donc produire plusieurs lignes. Il faut aplatir cette liste pour analyser chaque article sans perdre le lien avec le panier.
- json_normalize : json_normalize transforme des objets JSON, y compris leurs champs imbriqués, en colonnes tabulaires exploitables avec Pandas.


### Questions d'analyse
1. Quels champs identifient un produit de manière unique ?
2. Quel champ relie un panier à un utilisateur ?
3. Pourquoi la colonne `products` d'un panier est-elle complexe à analyser dans un tableau plat ?
4. Quel est le rôle d'une opération de normalisation (`json_normalize`) ?


In [8]:
# Explorer les champs et les types
print("Champs produits :", products_df.columns.tolist())
print("\nTypes produits :")
display(products_df.dtypes.astype(str).to_frame("type"))
print("\nStructure d'un panier :")
print(json.dumps(carts_payload["carts"][0], ensure_ascii=False, indent=2)[:3000])

print("\\nInterprétation : id est l'identifiant produit; userId relie un panier à son utilisateur. La normalisation rend les champs imbriqués plus faciles à examiner.")


Champs produits : ['id', 'title', 'description', 'category', 'price', 'discountPercentage', 'rating', 'stock', 'tags', 'brand', 'sku', 'weight', 'warrantyInformation', 'shippingInformation', 'availabilityStatus', 'reviews', 'returnPolicy', 'minimumOrderQuantity', 'images', 'thumbnail', 'dimensions.width', 'dimensions.height', 'dimensions.depth', 'meta.createdAt', 'meta.updatedAt', 'meta.barcode', 'meta.qrCode']

Types produits :


,type
id,int64
title,str
description,str
category,str
price,float64
discountPercentage,float64
rating,float64
stock,int64
tags,object
brand,str



Structure d'un panier :
{
  "id": 1,
  "products": [
    {
      "id": 162,
      "title": "Blue Frock",
      "price": 29.99,
      "quantity": 4,
      "total": 119.96,
      "discountPercentage": 12.13,
      "discountedTotal": 105.41,
      "thumbnail": "https://cdn.dummyjson.com/product-images/tops/blue-frock/thumbnail.webp"
    },
    {
      "id": 113,
      "title": "Generic Motorcycle",
      "price": 3999.99,
      "quantity": 3,
      "total": 11999.97,
      "discountPercentage": 12.1,
      "discountedTotal": 10547.97,
      "thumbnail": "https://cdn.dummyjson.com/product-images/motorcycle/generic-motorcycle/thumbnail.webp"
    },
    {
      "id": 122,
      "title": "iPhone 6",
      "price": 299.99,
      "quantity": 3,
      "total": 899.97,
      "discountPercentage": 6.69,
      "discountedTotal": 839.76,
      "thumbnail": "https://cdn.dummyjson.com/product-images/smartphones/iphone-6/thumbnail.webp"
    },
    {
      "id": 138,
      "title": "Baseball Ball",
   

## 4. Construire la fiche d'identité des sources

Complétez le tableau avec des éléments observés. Pour la fréquence, DummyJSON ne doit pas être interprété comme un flux de transactions en direct : vous interrogez une API de démonstration à la demande.


In [9]:
fiche_sources = pd.DataFrame([
    {"source":"Produits", "endpoint":"/products", "producteur":"DummyJSON", "format":"JSON", "volume_observe":len(products_df), "frequence":"à la demande", "usage_metier":"catalogue, prix, catégories", "limite":"données fictives, stock non transactionnel"},
    {"source":"Paniers", "endpoint":"/carts", "producteur":"DummyJSON", "format":"JSON imbriqué", "volume_observe":len(carts_df), "frequence":"à la demande", "usage_metier":"analyse de panier et quantités", "limite":"pas un journal réel de commandes"},
    {"source":"Utilisateurs", "endpoint":"/users", "producteur":"DummyJSON", "format":"JSON avec objets imbriqués", "volume_observe":len(users_df), "frequence":"à la demande", "usage_metier":"segmentation pédagogique", "limite":"profils fictifs ; minimiser les champs"},
    {"source":"Avis / commentaires", "endpoint":"/comments", "producteur":"DummyJSON", "format":"JSON", "volume_observe":int(pd.to_numeric(inventaire_df.loc[inventaire_df["ressource"] == "comments", "elements_reçus"], errors="coerce").fillna(0).iloc[0]), "frequence":"à la demande", "usage_metier":"analyse de texte", "limite":"commentaires associés aux posts, pas aux produits"}
])
display(fiche_sources)


,source,endpoint,producteur,format,volume_observe,frequence,usage_metier,limite
0,Produits,/products,DummyJSON,JSON,194,à la demande,"catalogue, prix, catégories","données fictives, stock non transactionnel"
1,Paniers,/carts,DummyJSON,JSON imbriqué,208,à la demande,analyse de panier et quantités,pas un journal réel de commandes
2,Utilisateurs,/users,DummyJSON,JSON avec objets imbriqués,208,à la demande,segmentation pédagogique,profils fictifs ; minimiser les champs
3,Avis / commentaires,/comments,DummyJSON,JSON,340,à la demande,analyse de texte,"commentaires associés aux posts, pas aux produits"


## 5. Évaluer les 5V

Pour chaque V, attribuez une appréciation **faible / moyen / élevé** et justifiez-la avec un fait observé. Ne donnez pas une appréciation sans argument.

- **Volume :** nombre d'enregistrements et taille des fichiers.
- **Vitesse :** fréquence de génération ou de mise à jour. Une API accessible ne signifie pas que les données sont du temps réel.
- **Variété :** ressources, formats, champs simples, objets et tableaux imbriqués.
- **Véracité :** cohérence, valeurs absentes, doublons, valeurs aberrantes et caractère fictif.
- **Valeur :** décisions que les données pourraient éclairer.


In [10]:
total_observe = int(pd.to_numeric(inventaire_df["elements_reçus"], errors="coerce").fillna(0).sum())
nb_ressources = len(inventaire_df)
analyse_5v = pd.DataFrame([
    {"V":"Volume","niveau":"faible","preuve_chiffree":f"{total_observe} éléments environ sur {nb_ressources} ressources dans cet instantané","justification":"Le jeu est de taille modeste et destiné à un exercice; il ne représente pas un entrepôt de données d'entreprise."},
    {"V":"Vitesse","niveau":"faible","preuve_chiffree":"1 requête à la demande; aucune fréquence de mise à jour garantie","justification":"L'API répond lorsqu'on l'interroge, mais ne fournit pas un flux de transactions en temps réel."},
    {"V":"Variété","niveau":"élevé","preuve_chiffree":f"{nb_ressources} ressources et plusieurs champs imbriqués (objets et listes)","justification":"Le jeu combine catalogue, paniers, utilisateurs et contenus; les structures JSON ne sont pas toutes plates."},
    {"V":"Véracité","niveau":"faible","preuve_chiffree":"Les paniers ne sont pas des ventes datées réelles; les dates de commande du projet sont synthétiques","justification":"Les données sont fictives et certaines relations ne correspondent pas à un système e-commerce réel; elles ne permettent pas de tirer des conclusions commerciales réelles."},
    {"V":"Valeur","niveau":"moyen","preuve_chiffree":"3 ressources centrales pour l'exercice : products, carts et users","justification":"Elles permettent de pratiquer des analyses de catalogue, de prix et de composition des paniers, mais la valeur décisionnelle réelle reste limitée par le caractère fictif et le faible volume."},
])
display(analyse_5v)

,V,niveau,preuve_chiffree,justification
0,Volume,faible,2959 éléments environ sur 8 ressources dans ce...,Le jeu est de taille modeste et destiné à un e...
1,Vitesse,faible,1 requête à la demande; aucune fréquence de mi...,"L'API répond lorsqu'on l'interroge, mais ne fo..."
2,Variété,élevé,8 ressources et plusieurs champs imbriqués (ob...,"Le jeu combine catalogue, paniers, utilisateur..."
3,Véracité,faible,Les paniers ne sont pas des ventes datées réel...,Les données sont fictives et certaines relatio...
4,Valeur,moyen,3 ressources centrales pour l'exercice : produ...,Elles permettent de pratiquer des analyses de ...


## 6. Usages métier, limites et risques

Pour chaque source principale, proposez au moins deux usages et deux limites. Exemples d'usages : analyser les catégories, repérer les paniers volumineux, comprendre la répartition des prix, préparer une segmentation illustrative. Exemples de limites : données fictives, absence d'horodatage transactionnel fiable, absence de paiements réels, volumes limités, absence de relation directe entre commentaires et produits.


In [11]:
usages_limites = pd.DataFrame([
    {"source":"Produits","usage_1":"Comparer les prix et les remises par catégorie","usage_2":"Étudier la répartition des stocks et des notes","limite_1":"Produits fictifs : pas représentatifs d'un vrai catalogue","limite_2":"Le stock observé n'est pas un historique de ventes"},
    {"source":"Paniers","usage_1":"Repérer les articles fréquemment réunis dans un même panier","usage_2":"Analyser les quantités et montants des paniers d'exemple","limite_1":"Paniers de démonstration, pas des commandes réelles","limite_2":"Pas de date de transaction fiable ni de paiement réel"},
    {"source":"Utilisateurs","usage_1":"Illustrer une segmentation selon les attributs disponibles","usage_2":"Relier les paniers à un profil utilisateur via userId","limite_1":"Profils fictifs, sans représentativité garantie","limite_2":"Champs personnels à minimiser et ne pas traiter comme de vraies données clients"},
    {"source":"Commentaires","usage_1":"Explorer du texte et illustrer une analyse de sentiment","usage_2":"Étudier des thèmes ou mots fréquents dans les commentaires","limite_1":"Commentaires associés aux posts, pas aux produits","limite_2":"Contenu fictif et sans lien fiable avec la satisfaction d'acheteurs réels"}
])
display(usages_limites)

,source,usage_1,usage_2,limite_1,limite_2
0,Produits,Comparer les prix et les remises par catégorie,Étudier la répartition des stocks et des notes,Produits fictifs : pas représentatifs d'un vra...,Le stock observé n'est pas un historique de ve...
1,Paniers,Repérer les articles fréquemment réunis dans u...,Analyser les quantités et montants des paniers...,"Paniers de démonstration, pas des commandes ré...",Pas de date de transaction fiable ni de paieme...
2,Utilisateurs,Illustrer une segmentation selon les attributs...,Relier les paniers à un profil utilisateur via...,"Profils fictifs, sans représentativité garantie",Champs personnels à minimiser et ne pas traite...
3,Commentaires,Explorer du texte et illustrer une analyse de ...,Étudier des thèmes ou mots fréquents dans les ...,"Commentaires associés aux posts, pas aux produits",Contenu fictif et sans lien fiable avec la sat...


## 7. Synthèse et livrable

### À remettre par binôme
1. Le tableau d'inventaire des ressources.
2. La fiche d'identité de trois sources au minimum.
3. L'évaluation argumentée des 5V.
4. Deux usages et deux limites pour chaque source principale.
5. Une conclusion de 8 à 12 lignes : quelles sources conserveriez-vous pour un projet d'analyse e-commerce et pourquoi ?

### Restitution orale
3 minutes par binôme. Présentez une source, un V dominant, une opportunité métier et une limite importante.

### Critères de réussite
- Données observées et chiffres sourcés dans la collecte.
- Distinction claire entre données API, données dérivées et hypothèses.
- Justifications argumentées, pas de simples mots-clés.
- Prise en compte du caractère fictif de DummyJSON.

**Transition vers l'Atelier 2 :** vous allez profiler un fichier tabulaire de lignes de panier avec Pandas et examiner les problèmes de qualité.


### Conclusion proposée (9 lignes)

Pour une analyse e-commerce, je conserverais d'abord les ressources `products`, `carts` et `users`.
`products` décrit le catalogue, les prix, les catégories et le stock observé.
`carts` permet d'étudier les articles et quantités réunis dans les paniers.
`users` permet de relier les paniers à des profils, uniquement pour une démonstration pédagogique.
Je garderais également `comments` si l'objectif est d'explorer du texte, en sachant qu'ils portent sur des posts.
Les autres ressources peuvent servir à comparer la variété des formats, mais elles sont moins centrales pour le cas commercial.
Avant toute analyse, je contrôlerais les identifiants, valeurs manquantes, doublons et relations entre ressources.
DummyJSON contient des données fictives et ses paniers ne sont pas des ventes historiques réelles.
Je n'utiliserais donc pas ces résultats pour prévoir le chiffre d'affaires ou prendre une décision commerciale réelle.